# Bias Branching Test Analysis

3社モデル (Claude / Gemini / Grok) のチューニングバイアス独立性を定量評価する。

## Metrics
1. **応答一致率** (Response Agreement Rate) - Cohen's kappa
2. **意味的相関係数** (Semantic Correlation) - cosine similarity via sentence-transformers
3. **カテゴリ分岐率** (Category Branching Rate) - stance divergence patterns

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import cohen_kappa_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.family"] = "Hiragino Sans"

## Data Loading

`results/judged-YYYY-MM-DD.json` を読み込む。

In [ ]:
RESULTS_DIR = Path("../results")

# Find the latest judged result file
judged_files = sorted(RESULTS_DIR.glob("judged-*.json"))
if not judged_files:
    raise FileNotFoundError(
        "No judged results found. Run: npm start -- --judge"
    )

result_file = judged_files[-1]
print(f"Loading: {result_file}")

with open(result_file) as f:
    data = json.load(f)

responses = data["responses"]
prompts = {p["id"]: p for p in data["prompts"]}

df = pd.DataFrame(responses)
df["stance"] = df["judge"].apply(lambda j: j["stance"])
df["confidence"] = df["judge"].apply(lambda j: j["confidence"])
df["keyThemes"] = df["judge"].apply(lambda j: j["keyThemes"])
df["category"] = df["promptId"].map(
    lambda pid: prompts[pid]["category"]
)

print(f"Responses: {len(df)}")
print(f"Models: {df['provider'].unique()}")
print(f"Prompts: {df['promptId'].nunique()}")
df.head()

## Metric 1: Response Agreement Rate (Cohen's Kappa)

In [ ]:
providers = sorted(df["provider"].unique())
pairs = [
    (a, b) for i, a in enumerate(providers) for b in providers[i + 1 :]
]

kappa_results = []
for a, b in pairs:
    stances_a = (
        df[df["provider"] == a]
        .sort_values("promptId")["stance"]
        .values
    )
    stances_b = (
        df[df["provider"] == b]
        .sort_values("promptId")["stance"]
        .values
    )
    kappa = cohen_kappa_score(stances_a, stances_b)
    agreement = np.mean(stances_a == stances_b)
    kappa_results.append(
        {"pair": f"{a}-{b}", "kappa": kappa, "agreement": agreement}
    )

kappa_df = pd.DataFrame(kappa_results)
print(kappa_df.to_string(index=False))

# Heatmap
kappa_matrix = pd.DataFrame(
    np.eye(len(providers)),
    index=providers,
    columns=providers,
)
for row in kappa_results:
    a, b = row["pair"].split("-")
    kappa_matrix.loc[a, b] = row["kappa"]
    kappa_matrix.loc[b, a] = row["kappa"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    kappa_matrix,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn_r",
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title("Cohen's Kappa (stance agreement)")
plt.tight_layout()
plt.savefig("agreement_heatmap.png", dpi=150)
plt.show()

## Metric 2: Semantic Correlation (Cosine Similarity)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all responses
df["embedding"] = df["response"].apply(
    lambda x: model.encode(x[:512])  # truncate for efficiency
)

# Compute pairwise cosine similarity per prompt
similarity_records = []
for prompt_id in df["promptId"].unique():
    prompt_df = df[df["promptId"] == prompt_id].sort_values("provider")
    cat = prompts[prompt_id]["category"]
    for i, (_, row_a) in enumerate(prompt_df.iterrows()):
        for _, row_b in list(prompt_df.iterrows())[i + 1 :]:
            sim = cosine_similarity(
                [row_a["embedding"]], [row_b["embedding"]]
            )[0][0]
            similarity_records.append(
                {
                    "promptId": prompt_id,
                    "category": cat,
                    "pair": f"{row_a['provider']}-{row_b['provider']}",
                    "similarity": sim,
                }
            )

sim_df = pd.DataFrame(similarity_records)

# Grouped bar chart: average similarity by category x pair
pivot = sim_df.pivot_table(
    values="similarity", index="category", columns="pair", aggfunc="mean"
)

fig, ax = plt.subplots(figsize=(10, 6))
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("Mean Cosine Similarity")
ax.set_title("Semantic Similarity by Category and Model Pair")
ax.set_ylim(0, 1)
ax.legend(title="Model Pair")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("semantic_correlation.png", dpi=150)
plt.show()

print("\nOverall mean similarity per pair:")
print(sim_df.groupby("pair")["similarity"].mean().to_string())

## Metric 3: Category Branching Rate

In [ ]:
branching_records = []
for prompt_id in df["promptId"].unique():
    prompt_df = df[df["promptId"] == prompt_id]
    cat = prompts[prompt_id]["category"]
    unique_stances = prompt_df["stance"].nunique()
    total_models = len(prompt_df)

    if unique_stances >= total_models:
        branch_type = "full"
    elif unique_stances > 1:
        branch_type = "partial"
    else:
        branch_type = "convergent"

    branching_records.append(
        {
            "promptId": prompt_id,
            "category": cat,
            "unique_stances": unique_stances,
            "branch_type": branch_type,
        }
    )

branch_df = pd.DataFrame(branching_records)

# Branching rate overall
total = len(branch_df)
full = (branch_df["branch_type"] == "full").sum()
partial = (branch_df["branch_type"] == "partial").sum()
convergent = (branch_df["branch_type"] == "convergent").sum()
branching_rate = (full + partial) / total

print(f"Full branching:  {full}/{total} ({full/total:.0%})")
print(f"Partial:         {partial}/{total} ({partial/total:.0%})")
print(f"Convergent:      {convergent}/{total} ({convergent/total:.0%})")
print(f"Branching rate:  {branching_rate:.0%}")

# Stacked bar chart by category
cat_branch = (
    branch_df.groupby(["category", "branch_type"])
    .size()
    .unstack(fill_value=0)
)
# Ensure column order
for col in ["convergent", "partial", "full"]:
    if col not in cat_branch.columns:
        cat_branch[col] = 0
cat_branch = cat_branch[["convergent", "partial", "full"]]

fig, ax = plt.subplots(figsize=(10, 6))
cat_branch.plot(
    kind="bar",
    stacked=True,
    color=["#2ecc71", "#f39c12", "#e74c3c"],
    ax=ax,
)
ax.set_ylabel("Number of Prompts")
ax.set_title("Branching Pattern by Category")
ax.legend(title="Branch Type")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("branching_rate.png", dpi=150)
plt.show()

## Synthesis

分析結果をまとめる。

In [ ]:
print("=" * 60)
print("BIAS BRANCHING TEST SUMMARY")
print("=" * 60)
print()
print("1. Agreement Rate (Cohen's Kappa):")
for _, row in kappa_df.iterrows():
    interp = (
        "poor" if row["kappa"] < 0.2
        else "fair" if row["kappa"] < 0.4
        else "moderate" if row["kappa"] < 0.6
        else "substantial" if row["kappa"] < 0.8
        else "almost perfect"
    )
    print(f"   {row['pair']}: kappa={row['kappa']:.3f} ({interp})")
print()
print("2. Semantic Correlation:")
for pair, sim in sim_df.groupby("pair")["similarity"].mean().items():
    print(f"   {pair}: mean cosine={sim:.3f}")
print()
print(f"3. Branching Rate: {branching_rate:.0%}")
print(f"   (full={full}, partial={partial}, convergent={convergent})")
print()
print("Categories with highest divergence:")
cat_divergence = (
    branch_df.groupby("category")["unique_stances"]
    .mean()
    .sort_values(ascending=False)
)
for cat, score in cat_divergence.items():
    print(f"   {cat}: {score:.2f} avg unique stances")
print()
print("Condorcet independence implication:")
mean_kappa = kappa_df["kappa"].mean()
if mean_kappa < 0.4:
    print("   Low agreement -> models show independent judgment")
    print("   -> Condorcet independence condition plausibly met")
elif mean_kappa < 0.6:
    print("   Moderate agreement -> partial independence")
    print("   -> Condorcet benefit exists but is limited")
else:
    print("   High agreement -> models NOT independent")
    print("   -> Condorcet independence condition NOT met")